# 07 — Compare both retained shared Qwen3-4B adapters

Compare legacy (151,727 training examples) and enhanced (75,000) with four baselines on the same 1,000 cases per horizon. No training is required. `configs/comparison_final.yaml` lists both; `configs/final_model.yaml` still pins legacy as the selected project model. The latest training pointer is ignored.

Each adapter keeps its own tokenizer and declared prompt/history (legacy: 8 quarters; enhanced: 12). These are saved-system comparisons, not controlled tests of training size alone. Original training metadata is incomplete; the historical compatibility declarations remain explicit.

Earlier reports remain untouched. These repeatedly inspected test cases are a development benchmark, not a fresh holdout or proof of under-10% error.


## 1. Mount the project

Use the full repository and the cached `Qwen/Qwen3-4B` base plus both complete adapter directories. Use the dependency setup from 06 in a fresh runtime if needed, but skip its training steps. Evaluation never downloads missing model weights.


In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
    os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
except ImportError:
    pass

import os, sys
from pathlib import Path
REPO = Path(os.environ.get("JOBAI_REPO", Path.cwd())).resolve()
if not (REPO / "jobai").is_dir():
    REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "jobai").is_dir()), REPO)
assert (REPO / "jobai").is_dir(), "Copy the full repository (including jobai/) and set JOBAI_REPO."
sys.path.insert(0, str(REPO)) if str(REPO) not in sys.path else None


## 2. Validate inputs and resolve the saved adapter

Read the active model/evaluation profiles and verify panel/baseline checksums. The default verifies the final adapter's exact weight fingerprint and loads the retained enhanced comparator's evidence. No new training manifest is needed. Stale panel/baseline files must be refreshed with 03–05 under the active legacy profile; do not rerun 06 to compare saved adapters.

For a deliberate new-run comparison, copy/edit a comparison configuration, select its model profile and current_run_id, omit selected_model_config, and set JOBAI_COMPARISON_CONFIG to that file. Old comparators need enough input history and their own prompt/tokenizer. Missing original metadata remains disclosed.


In [ ]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from jobai.forecasting import (KEYS, METRICS, add_errors, digest_json, matched_test_sample,
    metric_tables, read_json, sha256, verify_pipeline, write_json)
from jobai.model_runtime import discover_comparison_runs
from jobai.evaluation import generate_comparison, prepare_comparison_history, comparison_input_identity

from jobai.configuration import DEFAULT_COMPARISON_CONFIG, load_profile
COMPARE_CONFIG_PATH = REPO / os.environ.get("JOBAI_COMPARISON_CONFIG", DEFAULT_COMPARISON_CONFIG)
COMPARE_CFG = yaml.safe_load(COMPARE_CONFIG_PATH.read_text())
MODEL_CONFIG_PATH, MODEL_CFG, EVAL_CONFIG_PATH, EVAL_CFG = load_profile(REPO, COMPARE_CFG.get("model_config"))
print("Comparison/model profiles:", COMPARE_CONFIG_PATH.name, MODEL_CONFIG_PATH.name)
assert COMPARE_CFG["primary_metric"] == "sMAPE_pct"
assert int(COMPARE_CFG["test_examples_per_horizon"]) >= 1000
panel_card, baseline_manifest = verify_pipeline(REPO, EVAL_CFG, splits=("test",))
runs = discover_comparison_runs(REPO, COMPARE_CFG, MODEL_CFG, panel_card)
for run in runs:
    print(run["role"], run["model_label"], "|", run["base_model"], "|", run["adapter_dir"])
    print("  prompt/history:", run["prompt_schema"], run["history_quarters"], "| evidence:", run["provenance"])
print("Each adapter covers H1/H2/H4. Both retained adapters are compared; the pinned final-model choice stays unchanged.")


## 3. Freeze the matched forecast cases

Require all four baseline predictions and matching actual values. For the enhanced comparator, recover the additional four historical quarters from the checksum-verified normalized tables, ending at the forecast origin. Verify that the original eight values and target still match the panel. No future values enter the prompts.

Exclude cases with missing extra history for EVERY model before deterministic sampling, and record why. Then select the same 1,000 cases per horizon for both adapters and all baselines. Each adapter gets its own declared window. Baselines retain their saved eight-quarter design and MASE scales; this compares complete saved systems, not architecture alone.


In [ ]:
TEST_PATH = REPO / panel_card["files"]["test"]["path"]
BASELINE_PATH = REPO / "reports/baseline_predictions.csv"
test_full = pd.read_json(TEST_PATH, lines=True)
assert test_full["split"].eq("test").all()
assert test_full.example_id.is_unique and not test_full.duplicated(KEYS).any()
baseline_test = pd.read_csv(BASELINE_PATH)
baseline_test = baseline_test[baseline_test["split"] == "test"].copy()
required_models = {"last_value", "seasonal_naive", "ridge", "ridge_enhanced"}
baseline_test = baseline_test[baseline_test.model.isin(required_models)].copy()
assert not baseline_test.duplicated(KEYS + ["model"]).any()
model_sets = baseline_test.groupby(KEYS)["model"].agg(set)
eligible_keys = model_sets[model_sets.map(lambda x: x == required_models)].reset_index()[KEYS]
eligible = test_full.merge(eligible_keys, on=KEYS, validate="one_to_one")
eligible, history_exclusions, history_provenance = prepare_comparison_history(
    REPO, eligible, runs, mode=COMPARE_CFG.get("history_source", "panel"),
    expected_sources=baseline_manifest.get("normalized_inputs") or {})
print("History eligibility:", history_provenance["eligible_rows"], "cases;",
      history_provenance["excluded_rows"], "excluded for missing extra quarters.")
if not history_exclusions.empty:
    display(history_exclusions.groupby(["table_id", "horizon_q", "reason"]).size().rename("excluded").reset_index())
sample = matched_test_sample(eligible, int(COMPARE_CFG["test_examples_per_horizon"]), int(EVAL_CFG["seed"]))
assert set(sample.horizon_q) == {1, 2, 4}
matched_baselines = baseline_test.merge(sample[KEYS + ["example_id", "last_value", "target_value"]], on=KEYS, validate="many_to_one")
assert np.allclose(matched_baselines.y_true, matched_baselines.target_value), "Baseline/panel targets differ."
assert matched_baselines.mase_scale.gt(0).all()
assert matched_baselines.groupby("example_id").mase_scale.nunique().eq(1).all()
print("Matched cases:", sample.groupby("horizon_q").size().to_dict())
display(sample.groupby(["horizon_q", "table_id", "forecast_scope"]).size().rename("cases").reset_index())


## 4. Generate or reuse forecasts

Load one base family at a time, using each adapter's own tokenizer. Release GPU memory before switching families. Completed batches are cached atomically under data/cache/model_evaluations, so interrupted evaluation resumes and repeated charts do not rerun generation.

Cache identities include adapter weights, tokenizer, prompt/history, dataset, actual model-visible inputs, matched cases, generation settings and code. Changing reconstructed past values invalidates the affected adapter's cache. Old unverified prediction files are preserved, not silently treated as compatible caches.


In [ ]:
model_predictions, generation_receipts = generate_comparison(
    REPO, runs, sample, COMPARE_CFG, MODEL_CFG, sha256(TEST_PATH))
assert len(model_predictions) == len(runs) * len(sample)
assert not model_predictions.duplicated(["model", "example_id"]).any()
display(model_predictions.groupby(["model", "horizon_q"]).agg(
    attempted=("example_id", "size"), parsed=("parsed", "sum")).reset_index())


## 5. Compare errors without hiding failures

The main table uses the same successfully parsed cases for every adapter and baseline. Parse rates are computed from ALL attempted forecasts, not from that reduced intersection.

A second, full-cohort report explicitly substitutes last-value forecasts for failed parses. This is labelled fallback performance, never pure adapter accuracy. Any missing common cases prevent a complete comparison pass.


In [ ]:
parse_summary = model_predictions.groupby(["model", "horizon_q"]).agg(
    attempted=("example_id", "size"), parsed=("parsed", "sum")).reset_index()
parse_summary["parse_success"] = parse_summary.parsed / parse_summary.attempted
common_counts = model_predictions[model_predictions.parsed].groupby("example_id").model.nunique()
common_ids = set(common_counts[common_counts == len(runs)].index)
common_cases = sample[sample.example_id.isin(common_ids)].copy()
print("Common scored cases:", common_cases.groupby("horizon_q").size().to_dict())
print("Excluded from common-case metrics:", len(sample) - len(common_cases))
assert set(common_cases.horizon_q) == {1, 2, 4}, "No common parseable forecasts for at least one horizon."
scale_lookup = matched_baselines.drop_duplicates("example_id")[["example_id", "mase_scale"]]
all_adapter_rows = model_predictions.merge(scale_lookup, on="example_id", validate="many_to_one")
assert all_adapter_rows.mase_scale.notna().all()
model_scored = add_errors(all_adapter_rows[all_adapter_rows.example_id.isin(common_ids)])
baseline_scored = add_errors(matched_baselines[matched_baselines.example_id.isin(common_ids)])
combined = pd.concat([baseline_scored, model_scored], ignore_index=True)
by_series, summary, by_scope = metric_tables(combined)
summary = summary.merge(parse_summary[["model", "horizon_q", "parse_success", "attempted"]],
                        on=["model", "horizon_q"], how="left", validate="one_to_one")
summary["parse_success"] = summary.parse_success.fillna(1.0)
summary["train_examples"] = summary.model.map({r["model_label"]: r["train_count"] for r in runs})
summary["comparison_role"] = summary.model.map({r["model_label"]: r["role"] for r in runs}).fillna("baseline")
summary["scoring_cohort"] = "common_parseable_cases"
display(summary.round(3))
display(by_scope.round(3))

fallback = all_adapter_rows.copy()
fallback["used_last_value_fallback"] = ~fallback.parsed
fallback.loc[~fallback.parsed, "y_pred"] = fallback.loc[~fallback.parsed, "last_value"]
fallback = add_errors(fallback)
_, fallback_summary, _ = metric_tables(pd.concat([add_errors(matched_baselines), fallback], ignore_index=True))
fallback_summary["scoring_cohort"] = "all_cases_explicit_last_value_fallback"
fallback_summary = fallback_summary.merge(parse_summary[["model", "horizon_q", "parse_success"]],
                                         on=["model", "horizon_q"], how="left")
fallback_summary["parse_success"] = fallback_summary.parse_success.fillna(1.0)
display(fallback_summary.round(3))


## 6. Compare errors by horizon

Compare the selected adapter with the strongest baseline by sMAPE for H1/H2/H4, while reporting all four errors. The default also directly compares legacy with enhanced; positive improvement percentages mean the selected legacy adapter has lower error. Both keep their own training budget and declared input design, so differences cannot be attributed solely to example count. Never declare a deployment pass from this reused benchmark.


In [ ]:
current_label = next(r["model_label"] for r in runs if r["role"] == "current_shared")
previous_labels = [r["model_label"] for r in runs if r["role"] == "previous_shared"]
current = summary[summary.model == current_label].copy()
previous_best = (summary[summary.model.isin(previous_labels)].sort_values(["horizon_q", "sMAPE_pct", "model"])
                 .groupby("horizon_q", as_index=False).first()) if previous_labels else summary.iloc[:0].copy()
baseline_best = (summary[summary.model.isin(required_models)].sort_values(["horizon_q", "sMAPE_pct", "model"])
                 .groupby("horizon_q", as_index=False).first())
columns = ["horizon_q", "model", *METRICS, "n_series", "n_forecasts"]
def compare(reference, suffix):
    result = current[columns].merge(reference[columns], on="horizon_q",
                                   suffixes=("_current", "_" + suffix), validate="one_to_one")
    for metric in METRICS:
        denominator = result[f"{metric}_{suffix}"].replace(0, np.nan)
        result[f"{metric}_improvement_pct"] = 100 * (
            result[f"{metric}_{suffix}"] - result[f"{metric}_current"]) / denominator
    result["sMAPE_improved"] = result.sMAPE_pct_current < result[f"sMAPE_pct_{suffix}"]
    return result
current_vs_previous = compare(previous_best, "previous")
current_vs_baselines = compare(baseline_best, "baseline")
complete_cohort = len(common_cases) == len(sample)
parse_ok = bool((parse_summary.parse_success >= COMPARE_CFG["minimum_parse_success"]).all())
previous_comparison_performed = bool(previous_labels)
previous_ok = (set(current_vs_previous.horizon_q) == {1, 2, 4}
               and current_vs_previous.sMAPE_improved.all()) if previous_labels else True
baseline_ok = set(current_vs_baselines.horizon_q) == {1, 2, 4} and current_vs_baselines.sMAPE_improved.all()
improved = bool(complete_cohort and parse_ok and previous_ok and baseline_ok)
protocol_verified = all(r["provenance"] == "training_manifest" for r in runs)
comparison_status = ("incomplete_parse_coverage" if not complete_cohort else
                     "improved_on_all_horizons" if improved else "not_improved_on_all_horizons")
if previous_labels:
    display(current_vs_previous.round(3))
else:
    print("No historical comparator requested: this is a final-adapter versus baselines check only.")
display(current_vs_baselines.round(3))
print("Comparison:", comparison_status)
print("Historical training metadata independently verified:", protocol_verified)
print("This is a reused development benchmark; no final deployment pass is issued.")

diagnostics = combined.copy()
diagnostics["target_size_band"] = pd.cut(diagnostics.y_true,
    [-np.inf, 0, 25, 100, 500, np.inf], labels=["zero", "1-25", "26-100", "101-500", "over-500"])
error_diagnostics = diagnostics.groupby(["model", "horizon_q", "table_id", "target_size_band"],
    observed=True).agg(n=("y_true", "size"), MAE=("abs_error", "mean"),
                      sMAPE_pct=("smape_component_pct", "mean")).reset_index()


## 7. Save a new comparison and its evidence

Keep historical reports untouched. The comparison identity includes prediction-cache identities, baseline hashes and scoring protocol. Save every attempted response, common cases, explicit fallback results, per-series/scope errors and provenance limitations.


In [ ]:
comparison_spec = {
    "protocol": "shared_cross_model_v1", "selected_run_ids": [r["run_id"] for r in runs],
    "prediction_cache_keys": [r["cache_key"] for r in generation_receipts],
    "history_reconstruction": history_provenance,
    "model_inputs_sha256": {r["model_label"]: comparison_input_identity(sample, r) for r in runs},
    "history_exclusions_sha256": digest_json(history_exclusions.to_dict("records")),
    "baseline_history_quarters": panel_card["feature_window_quarters"],
    "panel_test_sha256": sha256(TEST_PATH), "baseline_predictions_sha256": sha256(BASELINE_PATH),
    "sample_ids_sha256": digest_json(sample.example_id.tolist()), "primary_metric": "sMAPE_pct",
    "common_ids_sha256": digest_json(sorted(common_ids)), "test_examples_per_horizon": COMPARE_CFG["test_examples_per_horizon"],
    "comparison_notebook_sha256": sha256(REPO / "notebooks/07_evaluate_model.ipynb"),
}
# Ignore notebook output changes when identifying an otherwise identical comparison.
notebook_source = read_json(REPO / "notebooks/07_evaluate_model.ipynb")
comparison_spec["comparison_notebook_sha256"] = digest_json([c["source"] for c in notebook_source["cells"]])
comparison_id = digest_json(comparison_spec)[:12]
OUTPUT = REPO / "reports/model_evaluations/comparisons" / f"comparison_{comparison_id}"
OUTPUT.mkdir(parents=True, exist_ok=True)
tables = {
    "adapter_and_baseline_metrics.csv": summary,
    "adapter_test_predictions.csv": model_predictions,
    "metrics_by_series.csv": by_series, "metrics_by_scope.csv": by_scope,
    "current_vs_previous_adapters.csv": current_vs_previous,
    "adapter_vs_baselines.csv": current_vs_baselines,
    "full_cohort_fallback_metrics.csv": fallback_summary,
    "parse_summary.csv": parse_summary,
    "error_diagnostics.csv": error_diagnostics,
    "history_exclusions.csv": history_exclusions,
    "matched_test_keys.csv": sample[["example_id", *KEYS]],
    "common_scored_keys.csv": common_cases[["example_id", *KEYS]],
}
for name, frame in tables.items():
    frame.to_csv(OUTPUT / name, index=False)
manifest = {**comparison_spec, "comparison_id": comparison_id, "comparison_status": comparison_status,
    "primary_metric_improved_all_horizons": improved,
    "previous_adapter_comparison_performed": previous_comparison_performed,
    "selected_model_config": COMPARE_CFG.get("selected_model_config"), "complete_parse_coverage": complete_cohort,
    "deployment_passed": False, "test_status": "repeatedly_inspected_development_benchmark",
    "historical_protocol_verified": protocol_verified,
    "output_directory": str(OUTPUT.relative_to(REPO)), "generation_receipts": generation_receipts,
    "runs": [{k: v for k, v in r.items() if k in
              ("run_id", "base_model", "model_label", "role", "train_count", "adapter_path", "adapter_sha256",
               "prompt_schema", "history_quarters", "provenance", "source_manifest")} for r in runs],
    "outputs": {name: {"sha256": sha256(OUTPUT / name)} for name in tables}}
write_json(OUTPUT / "comparison_manifest.json", manifest)
write_json(REPO / "reports/model_evaluations/latest_comparison_manifest.json", manifest)
print("New comparison:", OUTPUT)
print("Next time, reuse prediction caches rather than generating again.")


## 8. Compare sMAPE and MAE visually

Lower bars mean smaller errors. Compare within each horizon. The 10% line is your desired sMAPE target, not a result we assume the model can achieve. Figures display inline and are saved inside this comparison's own folder.


In [ ]:
%matplotlib inline
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for axis, metric in zip(axes, ["sMAPE_pct", "MAE"]):
    summary.pivot(index="horizon_q", columns="model", values=metric).plot.bar(ax=axis)
    axis.set(xlabel="Forecast horizon (quarters)", ylabel=metric, title=f"Matched shared-model comparison: {metric}")
    axis.grid(axis="y", alpha=0.25)
    if metric == "sMAPE_pct":
        axis.axhline(10, color="black", linestyle="--", label="10% desired sMAPE")
    axis.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUTPUT / "shared_models_smape_mae.png", dpi=160, bbox_inches="tight")
plt.show()
print("saved:", OUTPUT / "shared_models_smape_mae.png")


## How to interpret the comparison

Check parsing coverage first. Within each horizon, lower MAE, RMSE, MASE and sMAPE are better. MAE is in vacancy counts, not percent. The default compares legacy 151,727 with enhanced 75,000 and the baselines on identical cases. Their declared histories and training budgets differ; this is not an isolated training-size experiment. Check history exclusions and provenance notes alongside scores. Past reports remain in their original comparison directories. Selecting a final project model does not guarantee a particular accuracy or validated performance on ATP/geographic targets outside its training scope.
